# Falcon-H1 system-prompt search

Stock-only, deterministic prompt search. Candidate selection uses only the new development and validation batteries; the frozen final holdout remains untouched and is evaluated only after a prompt is selected. All raw generations and prompt-token costs are preserved.


In [ ]:
import os, subprocess, sys, time
from pathlib import Path
WORK = Path('/kaggle/working')
REPO = WORK / 'adtc-llm-limited-hardware'
OUT = WORK / 'falcon-prompt-search'
BRANCH = 'research/edge35-adaptive-streaming'
OUT.mkdir(parents=True, exist_ok=True)
def run(command, cwd=REPO, name='command.log'):
    log = OUT / name
    log.parent.mkdir(parents=True, exist_ok=True)
    print('STREAM', ' '.join(map(str, command)), flush=True)
    with log.open('a', encoding='utf-8', buffering=1) as handle:
        proc = subprocess.Popen([str(x) for x in command], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            stamp = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
            rendered = f'[{stamp}] {line}'
            print(rendered, end='', flush=True); handle.write(rendered); handle.flush()
        code = proc.wait()
        print('[{}] EXIT={}'.format(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), code), flush=True)
        if code: raise RuntimeError(f'command failed: {command}')
if not (REPO / '.git').exists():
    run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/qeinstein/adtc-llm-limited-hardware.git',str(REPO)], cwd=WORK, name='clone.log')
else:
    run(['git','-C',str(REPO),'fetch','origin',BRANCH], cwd=WORK, name='git-refresh.log')
    run(['git','-C',str(REPO),'checkout','-B',BRANCH,'origin/'+BRANCH], cwd=WORK, name='git-refresh.log')
run([sys.executable,'-m','pip','install','-q','-r','requirements-falcon-production.txt'], name='pip-base.log')
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], text=True, capture_output=True, check=False).stdout.strip()
if 'P100' in gpu:
    run([sys.executable,'-m','pip','install','-q','--upgrade','numpy<2'], name='numpy-p100.log')
    run([sys.executable,'-m','pip','install','-q','--upgrade','--force-reinstall','torch==2.6.0','--index-url','https://download.pytorch.org/whl/cu118'], name='torch-p100.log')
    run([sys.executable,'-m','pip','uninstall','-y','torchao','torchvision','torchaudio','bitsandbytes'], name='optional-uninstall.log')
    run([sys.executable,'-m','pip','install','-q','--upgrade','--force-reinstall','--no-deps','transformers==4.53.3','tokenizers==0.21.4','peft==0.15.2','accelerate==1.7.0'], name='hf-stack-final.log')
print({'gpu':gpu,'repo_sha':subprocess.run(['git','rev-parse','HEAD'],cwd=REPO,text=True,capture_output=True,check=True).stdout.strip()}, flush=True)


In [ ]:
candidate_file = os.environ.get('FALCON_PROMPT_CANDIDATES', 'docs/research/falcon_system_prompts_v4.json')
print('candidate file:', candidate_file, flush=True)
run([sys.executable,'-u','scripts/evaluate_falcon_prompt_candidates.py','--model','tiiuae/Falcon-H1-1.5B-Deep-Instruct','--revision','b6648636ddc906688974282de6e7a243395f5423','--candidates',candidate_file,'--development','docs/research/falcon_prompt_dev.json','--validation','docs/research/falcon_prompt_validation.json','--out',str(OUT)], name='prompt-search.log')
print('PROMPT_SEARCH_COMPLETE', (OUT/'prompt_search_results.json').read_text(), flush=True)
